# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The chosen focus for this task is **Lane-2: Content Opportunity Scoring**.

This is categorized as a **Ranking** task. Rather than utilizing a simple binary classification, the objective is to prioritize a limited queue of content pages. By assigning a priority score to each page, the items which pages may deserve earlier manual review can be surfaced.

**The intended output includes:**
- A standardized **Priority Score** for every page.
- A **Ranked Queue** to guide an editor's daily workflow.
- **Signal Drivers**: Brief explanations of why a page was flagged (e.g., 'Declining CTR despite high volume').
- **Human-in-the-loop**: The system provides suggestions; a human professional determines the final content action.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this initial experiment, the analysis is limited to content pieces with measurable search demand (at least 100 impressions over the last 90 days). Within this subset, a binary proxy labeled `review_priority_proxy` has been defined.

**The target is defined as:**
- **1 (Priority):** The `trend_direction` is confirmed as 'down'.
- **0 (Stable/Growth):** The `trend_direction` is 'stable' or 'up'.

**Rationale for this proxy:**
This identifies pages that are currently capturing significant traffic but are losing their performance momentum. By filtering for demand first and then predicting decline, the model can be used to rank pages based on how strongly they exhibit the characteristics of a potential review candidate.

*Note on Data Integrity: To prevent label leakage, the `trend_direction` column and any pre-computed health scores have been excluded from the feature set used for modeling.*

## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric has been defined as **Precision@50**.

**Significance of this metric:** A content team typically operates with a fixed weekly capacity for reviews (e.g., 50 pages). If the top 50 pages suggested by the model match the provisional observed-decline proxy, the system is deemed successful.

Performance is measured by comparing this output against a **Fixed-Rule Baseline** (e.g., a simple sort by raw impression loss). For the ML approach to be considered viable, it must outperform on held-out data by identifying more complex patterns of decline than the rule can capture.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis: The Content Page

To build a ranking system, the data must be analyzed through the lens of individual pieces of content.

*   **One row = One unique pseudonymized page.**
*   **Input Features:** Performance signals such as CTR, average position, and content age.
*   **Target:** The binary priority flag defined in the previous section.

The following code prepares this 'lane-specific' view, ensuring the resulting dataset contains rows representing distinct assets that can be actioned by an editor.

In [15]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd
from IPython.display import display

# The environment is configured to work in both Google Colab and local repositories.
if "google.colab" in sys.modules:
    repo_url = "https://github.com/pretom26/ml_internship_flyrankAI.git"
    repo_dir = Path("/content/ml_internship_flyrankAI")

    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, str(repo_dir)],
            check=True,
        )

    os.chdir(repo_dir)
else:
    # Search for the repository root by checking parent directories for the data file.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError(
            "The raw data file could not be located. "
            "Ensure the notebook is executed from within the repository."
        )

data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_direction",
]

# Validation for required schema columns.
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# Lane-specific filtering: Analysis is limited to pages with measurable search demand (at least 100 impressions).
lane_df = df.loc[df["impressions_90d"] >= 100, required_columns].copy()

# A provisional target/proxy is generated based on observed decline:
# 1 = downward trend
# 0 = stable or improving
lane_df["review_priority_proxy"] = (lane_df["trend_direction"].eq("down")).astype(int)

print(f"Total rows in the lane-specific dataframe (pages with demand): {len(lane_df):,}")
print(f"Total unique content items identified: {lane_df['content_id'].nunique():,}")
print(f"Total pseudonymized clients represented: {lane_df['client_id'].nunique():,}")
print("\nStructure verification: Each row represents a single pseudonymized content page.")

# Data preview utilizing shortened identifiers for readability.
preview = lane_df.head(8).copy()
preview["content_id"] = preview["content_id"].astype(str).str[:10] + "..."
preview["client_id"] = preview["client_id"].astype(str).str[:10] + "..."

display(preview)

Total rows in the lane-specific dataframe (pages with demand): 22,006
Total unique content items identified: 22,006
Total pseudonymized clients represented: 30

Structure verification: Each row represents a single pseudonymized content page.


,content_id,client_id,impressions_90d,ctr,avg_position,content_age_days,trend_direction,review_priority_proxy
0,content_30...,client_f36...,3803,0.76,10.6,187,down,1
1,content_a1...,client_4e0...,15320,0.05,20.3,445,down,1
2,content_9a...,client_7f2...,12581,0.09,36.5,141,down,1
3,content_33...,client_195...,11751,0.49,6.2,463,stable,0
4,content_d9...,client_3fd...,19140,0.13,44.0,263,down,1
5,content_d4...,client_f36...,3970,0.03,8.5,147,down,1
7,content_a6...,client_195...,1724,0.06,21.2,445,stable,0
8,content_5e...,client_620...,32574,0.09,46.0,90,down,1


In [16]:
# A summary of the target distribution is generated to evaluate label balance.
target_summary = (
    lane_df["review_priority_proxy"]
    .value_counts()
    .rename_axis("review_priority_proxy")
    .reset_index(name="page_count")
    .sort_values("review_priority_proxy", ascending=False)
)

target_summary["meaning"] = target_summary["review_priority_proxy"].map({
    1: "Provisional review candidate",
    0: "Does not meet the proxy definition",
})

target_summary["share_of_pages"] = (
    target_summary["page_count"] / len(lane_df) * 100
).round(1).astype(str) + "%"

display(
    target_summary[
        ["review_priority_proxy", "meaning", "page_count", "share_of_pages"]
    ]
)

,review_priority_proxy,meaning,page_count,share_of_pages
0,1,Provisional review candidate,13152,59.8%
1,0,Does not meet the proxy definition,8854,40.2%


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

While a fixed rule might prioritize older pages with low CTR and relatively strong search visibility, real-world content performance is **multi-dimensional**.

**ML provides an advantage due to:**
- **Non-linear interactions:** A slight drop in average position might be negligible for a new page but critical for a high-value legacy page. ML can weigh these signals concurrently.
- **Avoiding 'Edge Effects':** Rules create arbitrary thresholds (e.g., 99 vs 101 impressions). ML treats these as continuous gradients, leading to a smoother, more reliable ranking.
- **Scale and Nuance:** As additional signals—such as content age or seasonality—are incorporated, the number of 'if-statements' required to capture the pattern becomes unmanageable for manual maintenance.

**Actionable Output:**
This system functions as a **Decision Support Tool**. The ranked list is used to determine whether to refresh, prune, or monitor a page, effectively converting raw data into a prioritized work list.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w02_ml_task_framing.ipynb` — then submit your repo URL on the card. Done.